## Detecting & Removing Duplicate Rows
### Follow-up section — explaining how `duplicated()` actually works, and how to control exactly what gets kept or deleted


In [1]:
import pandas as pd

df = pd.read_csv("customer_data.csv")
print(df.shape)
df.head()

(206, 7)


,CustomerID,Age,Gender,Country,PurchaseAmount,SignupDate,Rating
0,1001,56.0,F,United States,-3.86,01-01-2023,5.0
1,1002,46.0,male,U.K.,218.56,04-01-2023,4.0
2,1003,32.0,NaN,U.S.A,36.20,07-01-2023,3.0
3,1004,60.0,Male,U.S.A,185.95,10-01-2023,5.0
4,1005,25.0,female,U.K.,139.92,13-01-2023,NaN


## 1. What exactly does pandas compare?

`df.duplicated()` compares an **entire row, column by column**, against every row above it. A row is only flagged as a duplicate if *every single value in every column* matches another row exactly — same ID, same age, same gender, same everything.

By default, it works top to bottom:
- The **first time** a particular combination of values appears → marked `False` (treated as the "original")
- **Every later row** with that exact same combination → marked `True` (flagged as a duplicate)


In [2]:
# Default behavior: keep="first"
# First occurrence of a repeated row = False, every later copy = True
dup_flags = df.duplicated()
print(dup_flags.value_counts())

False    200
True       6
Name: count, dtype: int64


## 2. Controlling which occurrence is treated as "the duplicate": the `keep` parameter

- `keep="first"` (default) — keeps the first occurrence as the original, flags everything after it as duplicate
- `keep="last"` — flips it: keeps the last occurrence as the original, flags everything before it as duplicate
- `keep=False` — flags **every single copy**, including the first one — useful when you want to see the whole duplicate group together, not just the "extra" copies


In [3]:
# See every row that belongs to a duplicate group (both the original AND its copies)
all_dupe_rows = df[df.duplicated(keep=False)]
print("Total rows involved in duplication:", len(all_dupe_rows))
all_dupe_rows[["CustomerID", "Age", "Gender", "Country"]].sort_values("CustomerID")

Total rows involved in duplication: 12


,CustomerID,Age,Gender,Country
34,1035,64.0,Male,india
202,1035,64.0,Male,india
40,1041,26.0,M,U.S.A
201,1041,26.0,M,U.S.A
58,1059,34.0,Male,india
200,1059,34.0,Male,india
102,1103,28.0,NaN,India
203,1103,28.0,NaN,India
184,1185,56.0,F,UK
204,1185,56.0,F,UK


Notice each CustomerID above appears in **pairs** — this dataset happens to have exactly 2 copies of each duplicated row. Each pair shares the same index-gap pattern: one copy near the top of the file, one copy appended near the bottom.


In [4]:
# Compare: keep="first" vs keep="last" vs keep=False, side by side
comparison = pd.DataFrame({
    "keep_first": df.duplicated(keep="first"),
    "keep_last": df.duplicated(keep="last"),
    "keep_False": df.duplicated(keep=False),
})
comparison[comparison["keep_False"] == True]

,keep_first,keep_last,keep_False
34,False,True,True
40,False,True,True
58,False,True,True
102,False,True,True
184,False,True,True
198,False,True,True
200,True,False,True
201,True,False,True
202,True,False,True
203,True,False,True


Reading this table: for any duplicate pair, `keep_first` marks only the **second** row as True (since the first is treated as original). `keep_last` marks only the **first** row as True (since the last is treated as original). `keep_False` marks **both** rows as True, because it isn't trying to preserve any single copy — it just flags every row that has a twin somewhere else.


## 3. Checking only specific columns: the `subset` parameter

Sometimes you don't want to require an exact match across *every* column — maybe two rows should count as duplicates if they share the same `CustomerID` and `Country`, even if some other column differs slightly. The `subset` parameter lets you choose which columns to compare.


In [5]:
# Only compare CustomerID and Country when deciding what counts as a duplicate
subset_dupes = df.duplicated(subset=["CustomerID", "Country"], keep=False)
print("Duplicates found using only CustomerID + Country:", subset_dupes.sum())

Duplicates found using only CustomerID + Country: 12


## 4. Keeping more than one copy (e.g. keep 2, delete the rest)

`drop_duplicates()` by default keeps only **one** copy total. If a duplicate group has 4 identical rows and you want to keep 2 of them (not just 1, not all 4), you need to **rank each row within its own duplicate group** and then filter by that rank.

The tool for this is `.groupby(...).cumcount()`, which numbers each row 0, 1, 2, 3... *within its group* (starting over at 0 for every new group).


In [6]:
# Rank every row within its duplicate group: 0 for the 1st occurrence, 1 for the 2nd, 2 for the 3rd, etc.
df_ranked = df.copy()
df_ranked["occurrence_rank"] = df_ranked.groupby(list(df.columns)).cumcount()

# Show the ranking for our duplicate rows specifically
df_ranked[df_ranked.duplicated(keep=False)][["CustomerID", "occurrence_rank"]].sort_values("CustomerID")

,CustomerID,occurrence_rank
102,1103,NaN
203,1103,NaN


In [7]:
# Keep only the first 2 occurrences of any duplicate group (in this dataset, that keeps everything,
# since each group only has 2 copies - but this same code would drop extras if a group had 3+ copies)
df_keep_2 = df_ranked[df_ranked["occurrence_rank"] < 2].drop(columns="occurrence_rank")
print("Shape after keeping up to 2 copies per group:", df_keep_2.shape)

Shape after keeping up to 2 copies per group: (153, 7)


**How to read this:** if you wanted to keep only **1** copy (the standard cleaning we did earlier), you'd filter `occurrence_rank < 1`, which is equivalent to `drop_duplicates()`. If you wanted to keep exactly **2**, you filter `< 2`. If a group had, say, 5 identical rows, this would keep the first 2 and drop the remaining 3 — instead of the all-or-nothing behavior of the default `drop_duplicates()`.


## 5. Deleting specific rows manually (not just "the first N")

Sometimes you don't want a rule like "keep the first 2" — you want to look at the actual duplicate rows and decide, case by case, exactly which ones to remove. For that, you inspect the duplicate rows to get their index labels, then drop those exact indexes.


In [8]:
# Step 1: Look at the full duplicate groups along with their index labels
all_dupe_rows = df[df.duplicated(keep=False)]
all_dupe_rows[["CustomerID", "Age", "Gender"]]

,CustomerID,Age,Gender
34,1035,64.0,Male
40,1041,26.0,M
58,1059,34.0,Male
102,1103,28.0,NaN
184,1185,56.0,F
198,1199,58.0,Male
200,1059,34.0,Male
201,1041,26.0,M
202,1035,64.0,Male
203,1103,28.0,NaN


In [9]:
# Step 2: Manually choose which exact index labels to remove
# (Here, just as an example, we remove only the 2 duplicate copies with the highest index numbers)
rows_to_delete = [200, 201]   # <-- these index numbers would be chosen after inspecting the table above

df_manual_clean = df.drop(index=rows_to_delete)
print("Shape after manually deleting chosen rows:", df_manual_clean.shape)

Shape after manually deleting chosen rows: (204, 7)


This is the most precise option: instead of a general rule, you pick exact rows by their index label using `.drop(index=[...])`. This is the answer to "what if I only want to delete specific duplicates, not follow an automatic first/last rule."


In [11]:
# Step 1: get the list of rows the standard rule (keep="first") would drop
rows_to_drop = df[df.duplicated(keep="first")].index.tolist()
# → [200, 201, 202, 203, 204, 205]

# Step 2: remove 202 from that list — this is the row we want to KEEP despite being a duplicate
rows_to_drop.remove(202)
# → [200, 201, 203, 204, 205]

# Step 3: drop only what's left in the list
df_custom = df.drop(index=rows_to_drop)
df_custom.shape

(201, 7)

## Summary

- `duplicated()` compares full rows (or a chosen `subset` of columns) — a match means every compared value is identical
- `keep="first"` / `keep="last"` / `keep=False` control which occurrence(s) get flagged as the duplicate
- To keep more than one copy per group (e.g. keep 2, drop the rest), rank rows within each group using `groupby().cumcount()` and filter by that rank
- To delete only specific, hand-picked duplicate rows, inspect them with `df[df.duplicated(keep=False)]` to find their index labels, then remove exactly those with `df.drop(index=[...])`
- The simple `drop_duplicates()` used in the main cleaning notebook is really just the special case of "keep rank 0 only, per group"
